In [0]:
# ============================================================
# 03C_dim_merge.ipynb
# Mục tiêu:
# - Merge dim_content_seed + dim_content_new_labels
# - Create final dim_content table
# ============================================================

from pyspark.sql import functions as F

In [0]:
# CONFIG
CATALOG = "workspace"
SCHEMA = "customer360"
DB = f"{CATALOG}.{SCHEMA}"

TABLE_DIM_SEED = f"{DB}.dim_content_new_seed"
TABLE_DIM_CONTENT = f"{DB}.dim_content"

PATH_LLM_LABELS = "/Volumes/workspace/customer360/c360_volume/dim_content_new_labels.parquet"

In [0]:
# LOAD TABLES
seed = spark.table(TABLE_DIM_SEED)
llm_labels = spark.read.parquet(PATH_LLM_LABELS)

In [0]:
# Ensure same schema order
seed = seed.select("content", "canonical_content", "category")
llm_labels = llm_labels.select("content", "canonical_content", "category")

In [0]:
# MERGE FINAL DIM
dim_content = (
    seed
    .unionByName(llm_labels)
    .dropDuplicates(["content"])
)

# SAVE TABLE
dim_content.write.mode("overwrite").saveAsTable(TABLE_DIM_CONTENT)

In [0]:
# Check results
print("dim_content rows:", spark.table(TABLE_DIM_CONTENT).count())

print("\nSample dim_content:")
spark.table(TABLE_DIM_CONTENT).show(20, truncate=False)

print("\nCategory distribution:")
spark.table(TABLE_DIM_CONTENT).groupBy("category").count().orderBy(F.col("count").desc()).show(20, truncate=False)

dim_content rows: 700

Sample dim_content:
+------------------------------------+------------------------------------+------------+
|content                             |canonical_content                   |category    |
+------------------------------------+------------------------------------+------------+
|vtv                                 |vtv                                 |News        |
|penthouse                           |penthouse                           |K-Drama     |
|bolero                              |bolero                              |Music       |
|taxi driver                         |taxi driver                         |Action      |
|running man                         |running man                         |Reality show|
|thử thách thần tượng                |thử thách thần tượng                |Reality show|
|Việt Nam                            |Việt Nam                            |Sport       |
|giao hữu bóng đá việt nam           |giao hữu bóng đá việt nam    